# 🔍 04 - Model Explainability with SHAP

This notebook provides:
1. **Global feature importance** — which features matter most across all predictions
2. **Per-prediction explanations** — why specific transactions are flagged
3. **SHAP force plots** — individual prediction breakdowns

**Why explainability matters:**
Fraud analysts won't trust a black box. They need to know *why* a transaction was flagged.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import joblib
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from src.predict import FraudPredictor

# Load model and data
X_train = joblib.load('data/processed/X_train.pkl')
X_test = joblib.load('data/processed/X_test.pkl')
y_test = joblib.load('data/processed/y_test.pkl')

feature_names = list(X_train.columns)

predictor = FraudPredictor(
    model=joblib.load('models/xgboost.pkl'),
    feature_names=feature_names,
    threshold=0.5,
)

print('Model and data loaded.')

## 1. Global Feature Importance (SHAP)

In [ ]:
# Compute SHAP values on a sample
n_shap = 500
X_shap = X_test.sample(n_shap, random_state=42)

explainer = shap.TreeExplainer(predictor.model)
shap_values = explainer.shap_values(X_shap)

# For XGBoost binary classification, shap_values might be a list
if isinstance(shap_values, list):
    shap_vals = shap_values[1]  # Fraud class
else:
    shap_vals = shap_values

print(f'SHAP values computed for {n_shap} samples.')
print(f'Shape: {shap_vals.shape}')

In [ ]:
# Global feature importance bar plot
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
feat_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': mean_abs_shap
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(feat_importance)))
ax.barh(feat_importance['feature'], feat_importance['importance'], color=colors)
ax.set_xlabel('Mean |SHAP value|', fontsize=12)
ax.set_title('Global Feature Importance (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/processed/shap_global_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTop 10 most important features for fraud detection:')
for _, row in feat_importance.tail(10).iloc[::-1].iterrows():
    print(f'  {row["feature"]}: {row["importance"]:.4f}')

## 2. SHAP Summary Plot (Beeswarm)

In [ ]:
# SHAP beeswarm plot
plt.figure()
shap.summary_plot(shap_vals, X_shap, feature_names=feature_names,
                  show=False, max_display=15)
plt.title('SHAP Summary Plot — Feature Impact on Fraud Prediction', fontsize=14)
plt.tight_layout()
plt.savefig('data/processed/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretation:')
print('  - Red = high feature value, Blue = low feature value')
print('  - SHAP > 0 pushes prediction toward fraud')
print('  - SHAP < 0 pushes prediction toward legitimate')

## 3. Per-Prediction Explanations

Pick a specific flagged fraud transaction and explain it.

In [ ]:
# Find a correctly caught fraud
y_proba = predictor.model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

# Find a true positive (fraud correctly caught)
tp_idx = np.where((y_pred == 1) & (y_test.values == 1))[0]

if len(tp_idx) > 0:
    sample_idx = tp_idx[0]
    sample_txn = X_test.iloc[sample_idx].to_dict()
    
    result = predictor.predict_single(sample_txn, return_shap=True, X_background=X_train)
    
    print(f'\nTransaction #{sample_idx}')
    print(f'Fraud Probability: {result["fraud_probability"]:.4f}')
    print(f'Decision: {result["decision"]}')
    print(f'\nTop contributing features:')
    for feat in result['explanation']['top_features'][:5]:
        direction = '↑ FRAUD' if feat['impact'] == 'increases' else '↓ SAFE'
        print(f'  {feat["feature"]} = {feat["value"]:.4f} (SHAP: {feat["shap_value"]:+.4f}) {direction}')
else:
    print('No correctly caught fraud in test set.')

In [ ]:
# SHAP force plot for the same transaction
shap_sample = X_shap.iloc[[0]]
shap_val_sample = shap_vals[0]

plt.figure()
shap.force_plot(
    explainer.expected_value if not isinstance(explainer.expected_value, list) else explainer.expected_value[1],
    shap_val_sample,
    shap_sample,
    feature_names=feature_names,
    matplotlib=True,
    show=False,
)
plt.title('SHAP Force Plot — Individual Transaction Explanation', fontsize=12)
plt.tight_layout()
plt.savefig('data/processed/shap_force_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SHAP Dependence Plots

In [ ]:
# Dependence plots for top features
top_feats = ['V14', 'V4', 'V12', 'V10']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, feat in enumerate(top_feats):
    ax = axes[i // 2][i % 2]
    feat_idx = feature_names.index(feat)
    
    scatter = ax.scatter(
        X_shap[feat].values,
        shap_vals[:, feat_idx],
        c=y_test.iloc[X_shap.index].values,
        cmap='RdYlGn_r',
        alpha=0.5,
        s=10,
    )
    ax.set_xlabel(feat, fontsize=12)
    ax.set_ylabel(f'SHAP value for {feat}', fontsize=12)
    ax.set_title(f'{feat} Dependence', fontsize=13, fontweight='bold')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('SHAP Dependence Plots', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('data/processed/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Summary

**Key findings:**
- **V14** is the most important feature — highly negative values strongly predict fraud
- **V4** and **V12** are also major contributors
- The model correctly learns that extreme PCA values (far from 0) indicate fraud
- SHAP explanations can be served via the API to give analysts per-transaction reasoning

**Next step:** FastAPI deployment with SHAP explanations in the API response.